<a href="https://colab.research.google.com/github/vashirij/wildfire-tinyml-self-sufficiency/blob/main/Notebooks/07_Edge_Deployment_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 Edge Deployment Validation

## Objectives

- Load the optimized TinyML model.
- Simulate deployment on a resource-constrained edge device.
- Measure latency, memory footprint, throughput, and estimated energy.
- Evaluate continuous streaming inference.
- Generate deployment-ready metrics and figures.

> This notebook performs **virtual deployment validation** suitable for later migration to ESP32, STM32, Raspberry Pi Pico, or other TinyML hardware.


In [ ]:
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT=Path('/content/drive/MyDrive/WildfireProject')

DATA_PATH=PROJECT_ROOT/'data'/'simulated'/'realistic'/'realistic_environment_stream.csv'
MODEL_PATH=PROJECT_ROOT/'models'/'tinyml'/'tinyml_best_model.joblib'

RESULT_DIR=PROJECT_ROOT/'results'/'deployment'
FIG_DIR=PROJECT_ROOT/'figures'/'deployment'
REPORT_DIR=PROJECT_ROOT/'docs'/'deployment'

for d in [RESULT_DIR,FIG_DIR,REPORT_DIR]:
    d.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(DATA_PATH)
model=joblib.load(MODEL_PATH)

FEATURES=[
'temperature_c',
'humidity_percent',
'smoke_ppm',
'co_ppm',
'wind_speed_kmh'
]

X=df[FEATURES]
y=df['wildfire']
print(df.shape)


In [ ]:
DEVICE={
'platform':'ESP32-S3 (Virtual)',
'clock_mhz':240,
'ram_kb':512,
'flash_mb':16,
'battery_voltage':3.7
}

ENERGY_PER_INFERENCE_MJ=0.08
ENERGY_PER_TRANSMISSION_MJ=2.5
FIRE_THRESHOLD=0.30


In [ ]:
latencies=[]
predictions=[]
probabilities=[]
energy=[]

for _,row in X.iterrows():
    sample=pd.DataFrame([row],columns=FEATURES)

    t0=time.perf_counter()
    if hasattr(model,"predict_proba"):
        p=float(model.predict_proba(sample)[0,1])
    else:
        p=float(model.predict(sample)[0])

    pred=int(p>=FIRE_THRESHOLD)
    dt=(time.perf_counter()-t0)*1000

    latencies.append(dt)
    predictions.append(pred)
    probabilities.append(p)

    e=ENERGY_PER_INFERENCE_MJ
    if pred:
        e+=ENERGY_PER_TRANSMISSION_MJ
    energy.append(e)

deployment=pd.DataFrame({
'timestamp':df['timestamp'],
'prediction':predictions,
'probability':probabilities,
'latency_ms':latencies,
'energy_mj':energy,
'true_label':y
})
display(deployment.head())


In [ ]:
metrics={
'accuracy':accuracy_score(y,predictions),
'precision':precision_score(y,predictions,zero_division=0),
'recall':recall_score(y,predictions,zero_division=0),
'f1_score':f1_score(y,predictions,zero_division=0),
'average_latency_ms':float(np.mean(latencies)),
'maximum_latency_ms':float(np.max(latencies)),
'throughput_samples_per_second':1000/np.mean(latencies),
'average_energy_mj':float(np.mean(energy)),
'total_energy_mj':float(np.sum(energy)),
'model_size_kb':MODEL_PATH.stat().st_size/1024
}

summary=pd.DataFrame([metrics])
display(summary.round(4))


In [ ]:
deployment.to_csv(
RESULT_DIR/'week7_streaming_predictions.csv',
index=False
)

summary.to_csv(
RESULT_DIR/'week7_deployment_summary.csv',
index=False
)

plt.figure(figsize=(10,4))
plt.plot(deployment['latency_ms'])
plt.title('Inference Latency')
plt.ylabel('Latency (ms)')
plt.tight_layout()
plt.savefig(FIG_DIR/'latency.png',dpi=300)
plt.show()

plt.figure(figsize=(10,4))
plt.plot(np.cumsum(deployment['energy_mj']))
plt.title('Cumulative Energy Consumption')
plt.ylabel('Energy (mJ)')
plt.tight_layout()
plt.savefig(FIG_DIR/'energy.png',dpi=300)
plt.show()

cm=confusion_matrix(y,predictions)
plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.title('Confusion Matrix')
plt.colorbar()
plt.tight_layout()
plt.savefig(FIG_DIR/'confusion_matrix.png',dpi=300)
plt.show()


In [ ]:
report=pd.DataFrame([{
'virtual_device':DEVICE['platform'],
'ram_kb':DEVICE['ram_kb'],
'flash_mb':DEVICE['flash_mb'],
'model_size_kb':metrics['model_size_kb'],
'average_latency_ms':metrics['average_latency_ms'],
'throughput_samples_per_second':metrics['throughput_samples_per_second'],
'average_energy_mj':metrics['average_energy_mj'],
'accuracy':metrics['accuracy'],
'precision':metrics['precision'],
'recall':metrics['recall'],
'f1_score':metrics['f1_score']
}])

report.to_csv(REPORT_DIR/'edge_validation_report.csv',index=False)

print('='*70)
print('EDGE DEPLOYMENT VALIDATION COMPLETE')
print('='*70)
print('Virtual Device:',DEVICE['platform'])
print('Average latency (ms):',round(metrics['average_latency_ms'],4))
print('Throughput (samples/sec):',round(metrics['throughput_samples_per_second'],2))
print('Model size (KB):',round(metrics['model_size_kb'],2))
print('F1-score:',round(metrics['f1_score'],4))
